<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/3-2_llama2-moderation-chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>3.2-Create a Moderation System using LangChain.</h2>
    <h3>LLAMA 2 version</h3>
    <p>by <b>Pere Martra</b></p>
</div>

This notebook needs an environment with GPU. I'm using a A100 GPU but it can run with any 16GB GPU.

# How To Create a Moderation System Using LangChain & Hugging Face.

We are going to create a Moderation System based in two Models. The first Model  reads the User comments and answer them.

The second language Model receives the answer of the first model and identify any kind on negativity modifying if necessary the comment.

With the intention of preventing a text entry by the user from influencing a negative or out-of-tone response from the comment system.

In [ ]:
# === Colab dependency guard (bban4040) ===
# The langchain / langsmith stack upgrades transitive packages (requests,
# opentelemetry-*) past the exact versions Colab's preinstalled google
# packages pin (google-colab, google-adk, the otlp/gcp exporters), which
# prints noisy "pip's dependency resolver ... is incompatible" errors.
# We pin those families to the versions already installed so the installs
# below leave them untouched. PIP_CONSTRAINT is honored by every %pip call
# in this kernel. Harmless off Colab (nothing matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")


In [ ]:
#Install LangChain (1.x) and the HF integration packages.
%pip install -q langchain langchain-core langchain-classic langchain-huggingface
%pip install -q transformers accelerate

In [ ]:
%pip install -q huggingface_hub

In [ ]:
# === portable-setup (bban4040) ===
# Resolve the Hugging Face token from Colab Secrets (userdata), a local .env,
# or environment variables -- never prompt, so the notebook runs unattended.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
def get_secret(name, default=None):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name, default)
    return v.strip() if isinstance(v, str) else v
hf_key = get_secret('HF_TOKEN') or get_secret('HUGGINGFACEHUB_API_TOKEN')

In [ ]:
from huggingface_hub import login
if hf_key:
    login(token=hf_key)
else:
    print('No HF token found. Set HF_TOKEN in Colab Secrets / .env to load gated Llama-2.')

## Importing LangChain Libraries.
* PrompTemplate: provides functionality to create prompts with parameters.
* OpenAI:  To interact with the OpenAI models.
* LLMChain: To create chains, where the prompts or the results can pass from one step to another inside the chain.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
import torch
from torch import cuda

In [ ]:
#In a MAC Silicon the device must be 'mps'
# device = torch.device('mps') #to use with MAC Silicon
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

In [ ]:
device

##Load the Model .

In [ ]:
#You can try with any llama model, but you will need more GPU and memory as you
#increase the size of the model.
model_id = "meta-llama/Llama-2-7b-chat-hf"

In [ ]:
# begin initializing HF items, need auth token for these
model_config = transformers.AutoConfig.from_pretrained(
    model_id,
    token=hf_key
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    config=model_config,
    device_map='auto',
    token=hf_key
)
model.eval()
print(f"Model loaded on {device}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id,
                                          token=hf_key)


In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.1,
    #do_sample=False,
    top_p=0,
    #trust_remote_code=True,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.1,
    return_full_text=True,
    device_map='auto'
)

assistant_llm = HuggingFacePipeline(pipeline=pipe)

## Create the template for the first model called assistant.

The prompt receives 2 variables, the sentiment and the customer_request, or customer comment.

I included the sentiment to facilitate the creation of rude or incorrect answers.

In [ ]:
# Instruction how the LLM must respond the comments,
assistant_template = """
[INST]<<SYS>>You are {sentiment} assistant that responds to user comments,
using similar vocabulary than the user.
Stop answering text after answer the first user.<</SYS>>

User comment:{customer_request}[/INST]
assistant_response
"""

In [ ]:
#Create the prompt template to use in the Chain for the first Model.
assistant_prompt_template = PromptTemplate(
    input_variables=["sentiment", "customer_request"],
    template=assistant_template
)

Now we create a First Chain. Just chaining the assistant_prompt_template and the model. The model will receive the prompt generated with the prompt_template.

In [ ]:
output_parser = StrOutputParser()
assistant_chain = assistant_prompt_template | assistant_llm | output_parser

To execute the chain created it's necessary to call the .run method of the chain, and pass the variables necessaries.

In our case: customer_request and sentiment.

In [ ]:
#Support function to obtain a response to a user comment.
def create_dialog(customer_request, sentiment):
    #callint the .invoke method from the chain created Above.
    assistant_response = assistant_chain.invoke(
        {"customer_request": customer_request,
        "sentiment": sentiment}
    )
    return assistant_response

## Obtain answers from our first Model Unmoderated.

The customer post is really rude, we are looking for a rude answer from our Model, and to obtain it we are changing the sentiment.

In [ ]:
# This the customer request, or customer comment in the forum moderated by the agent.
# feel free to update it.
customer_request = """Your product is a piece of shit. I want my money back!"""

In [ ]:
# Our assistatnt working in 'nice' mode.
assistant_response=create_dialog(customer_request, "nice")
print(assistant_response)

In [ ]:
#Our assistant running in rude mode.
assistant_response = create_dialog(customer_request, "most rude assistant that exist")
print(assistant_response)

Okay, this answer needs some moderation! Fortunately, we are actively working on it!

## Moderator
Let's create the second moderator. It will receive the message generated previously and rewrite it if necessary.

In [ ]:
#The moderator prompt template
moderator_template = """
[INST]<<SYS>>You are the moderator of an online forum, you are strict and will not tolerate any negative comments.
You will receive an original comment and if it is impolite you must transform into polite.
Try to mantain the meaning when possible.<</SYS>>

Original comment: {comment_to_moderate}/[INST]
"""

# We use the PromptTemplate class to create an instance of our template that will use the prompt from above and store variables we will need to input when we make the prompt.
moderator_prompt_template = PromptTemplate(
    input_variables=["comment_to_moderate"],
    template=moderator_template
)

In [ ]:
moderator_llm = assistant_llm

In [ ]:
#We build the chain for the moderator.

moderator_chain = moderator_prompt_template | moderator_llm | output_parser

In [ ]:
assistant_response

In [ ]:
# To run our chain we use the .invoke() command
moderator_says = moderator_chain.invoke({"comment_to_moderate": assistant_response})

In [ ]:
print(moderator_says)

This answer is more polite that the one produce by the  **"rude" assistant**.

## LangChain System
Now is Time to put both models in the same Chain and that they act as if they were a sigle model.

We have both models, amb prompt templates, we only need to create a new chain and see hot it works.



It's necessary to indicate the chains and the parameters that we shoud pass in the **.run** method.

In [ ]:
assistant_moderated_chain = (
    {"comment_to_moderate":assistant_chain}
    |moderator_chain
)

Lets use our Moderating System!

In [ ]:
# We can now run the chain.
from langchain_core.tracers import ConsoleCallbackHandler
assistant_moderated_chain.invoke({"sentiment": "very rude", "customer_request": customer_request},
                                 config={'callbacks':[ConsoleCallbackHandler()]})

## Conclusions
As You can see how the moderator changes the answer of our assistant. The one produced by the moderator is by far more polite than the original response created by the rude assistant.
